<a href="https://colab.research.google.com/github/vineet-crypto/Distorted-Visual-Sequence-Pattern-Recognition-using-Deep-Learning/blob/main/notebook_VineetSinghania_24115162.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Distorted Visual Sequence Pattern Recognition using Deep Learning**

Many modern digital systems need to read text from images that are unclear, noisy, or intentionally distorted. In real-world situations, characters may overlap with each other, contain random noise, be partially hidden, or appear in unusual shapes and positions. These challenges make normal text recognition systems unreliable.


The dataset contains grayscale images along with their correct text labels for training. Each image contains a sequence of characters affected by different types of distortions such as:
* Background noise
* Overlapping symbols
* Blur and visual artifacts
* Shape deformation
* Occlusion and random patches
* Irregular spacing and alignment

The goal is to design a model that can learn meaningful visual patterns and accurately predict the correct ordered sequence of characters even under difficult conditions.

# Part 1: Environment & Dependency Setup

**Dependencies**: Unzips the dataset and installs the required library for calculating the error rate.

**AI Framework**: Imports the core deep learning tools to build and optimize your neural network.

**Data Pipeline**: Prepares the necessary modules for loading images, applying augmentations, and batching the data for training.

In [ ]:
!unzip -q cig_ps.zip
!pip install Levenshtein

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import pandas as pd
import numpy as np
from PIL import Image
import Levenshtein
from dataclasses import dataclass
from typing import Tuple, Union, List
from torch.optim import Optimizer

# Part 2: Hyperparameters & Configuration
**Model Settings**: Establishes the core training variables, including learning rate, batch size, epochs, target image dimensions, and active hardware (GPU/CPU).

**Dynamic Routing**: Automatically scans and sets the correct directory paths for the training and testing datasets.

**Text Encoding**: Creates the vocabulary mappings to translate the expected alphanumeric text characters into numerical IDs that the neural network can process.

In [ ]:
@dataclass
class TrainingConfig:
    batch_size: int = 32
    learning_rate: float = 0.0001
    epochs: int = 30
    image_h: int = 64
    image_w: int = 256
    compute_node: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the configuration settings
config = TrainingConfig()

def find_dataset_root(search_paths: list) -> str:
    """Scans a list of prefixes to find the dataset root directory."""
    for path in search_paths:
        if os.path.exists(os.path.join(path, "train-labels.csv")):
            return path
    return ""

folder_options = ["", "cig_ps/", "cig_data/"]
data_root = find_dataset_root(folder_options)

# Map final resource paths
train_csv_file = os.path.join(data_root, "train-labels.csv")
train_image_folder = os.path.join(data_root, "train_images")
test_image_folder = os.path.join(data_root, "test_images")

valid_characters = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"

# Dictionary comprehension: Index starts at 1 (0 is reserved for CTC blank)
char_2_idx = {character: index + 1 for index, character in enumerate(valid_characters)}
idx_2_char = {index: character for character, index in char_2_idx.items()}
total_model_classes = len(valid_characters) + 1

print(f"Hardware assigned: {config.compute_node}")
print(f"Dataset resolved to path: '{data_root if data_root else './'}'")

Hardware assigned: cuda
Dataset resolved to path: './'


# Part 3: Data Loading & Preprocessing Pipeline
**Dataset Engine**: Loads images as grayscale, standardizes sizes, and dynamically converts text labels into numerical sequences for training or couples images with filenames for testing.

**Batch Collator**: Packs batches together by stacking images, flattening text sequences, and tracking variable label lengths for sequence-based loss functions.

**Transform Filters**: Standardizes dimensions and normalizes pixels for both sets, adding subtle brightness and contrast shifts during training to improve robustness against artifacts.

In [ ]:
class TextRecognitionDataset(Dataset):
    """
    Custom PyTorch Dataset for loading images and encoding text labels for OCR.
    """
    def __init__(self, metadata_df: pd.DataFrame, image_folder: str,
                 image_transform=None, evaluation_mode: bool = False,
                 file_col: str = 'image', text_col: str = 'text'):
        self.metadata = metadata_df
        self.image_folder = image_folder
        self.image_transform = image_transform
        self.evaluation_mode = evaluation_mode
        self.file_col = file_col
        self.text_col = text_col

    def __len__(self) -> int:
        return len(self.metadata)

    def __getitem__(self, index: int) -> Union[Tuple[torch.Tensor, str], Tuple[torch.Tensor, torch.Tensor]]:
        # Retrieve row data
        row_data = self.metadata.iloc[index]
        file_name = row_data[self.file_col]
        full_path = os.path.join(self.image_folder, file_name)

        # Open image and convert to grayscale ('L')
        raw_image = Image.open(full_path).convert('L')

        # Apply augmentations/transformations
        if self.image_transform:
            raw_image = self.image_transform(raw_image)

        # If testing, labels aren't needed. Return image and filename.
        if self.evaluation_mode:
            return raw_image, file_name

        # For training, encode the text label into integers
        raw_text = str(row_data[self.text_col]).upper()

        # Using the char_2_idx mapping defined in the previous block
        encoded_sequence = [char_2_idx[char] for char in raw_text if char in char_2_idx]

        return raw_image, torch.tensor(encoded_sequence, dtype=torch.long)

def ctc_collate_batch(batch_data: List) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

    # Custom collation to handle variable-length text labels for CTC Loss.
    image_list, label_list = zip(*batch_data)

    # Stack images into a single batch tensor
    stacked_images = torch.stack(image_list, 0)

    # Record the length of each label before concatenation
    sequence_lengths = torch.tensor([len(label) for label in label_list], dtype=torch.long)

    # Concatenate all targets into a single 1D tensor (Required by PyTorch CTCLoss)
    concatenated_targets = torch.cat(label_list)

    return stacked_images, concatenated_targets, sequence_lengths

# Notice we are using 'config' from our previous configuration class
training_augmentations = T.Compose([
    T.Resize((config.image_h, config.image_w)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

validation_augmentations = T.Compose([
    T.Resize((config.image_h, config.image_w)),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

# Part 4: CRNN Model Architecture

**Visual Extraction**: Employs a Convolutional Neural Network (CNN) to pull specific shapes and features directly from the distorted image inputs.  

**Sequence Processing**: Uses a Bidirectional LSTM to read those extracted features from both directions, understanding the sequential order of the text.  

**Character Decoding**: Maps the final sequential data to the specific alphanumeric characters in your defined vocabulary to form the predicted word.

In [ ]:
class VisionSequenceReader(nn.Module):
    """
    CRNN architecture combining a Convolutional Feature Extractor
    with a Bidirectional LSTM for sequence modeling.
    """
    def __init__(self, total_classes: int, rnn_hidden_dim: int = 256):
        super().__init__()

        # 1. Feature Extraction Pipeline (CNN)
        # Using inplace=True for ReLU to optimize memory usage
        self.feature_extractor = nn.Sequential(
            # Conv Block 1
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Conv Block 2
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Conv Block 3
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),

            # Conv Block 4
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),

            # Conv Block 5 (Final feature map generation)
            nn.Conv2d(512, 512, kernel_size=2, stride=1, padding=0),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )

        # 2. Sequence Transformation
        # Given an input height of 64, the pooling layers reduce the height to 3.
        cnn_output_features = 512 * 3
        self.sequence_projection = nn.Linear(cnn_output_features, rnn_hidden_dim)

        # 3. Sequence Modeling (RNN)
        self.recurrent_layers = nn.LSTM(
            input_size=rnn_hidden_dim,
            hidden_size=rnn_hidden_dim,
            bidirectional=True,
            num_layers=2,
            batch_first=False,
            dropout=0.3
        )

        # 4. Final Classification Head
        # Multiplied by 2 because the LSTM is bidirectional
        self.classifier_head = nn.Linear(rnn_hidden_dim * 2, total_classes)

    def forward(self, image_batch: torch.Tensor) -> torch.Tensor:
        # Step 1: Extract visual feature maps
        visual_features = self.feature_extractor(image_batch)

        # Step 2: Unpack dimensions (Batch, Channels, Height, Width)
        batch_size, channels, height, width = visual_features.size()

        # Step 3: Collapse Channels and Height into a single feature vector per timestep (width)
        # Shape becomes: (Batch, Features, Sequence_Length)
        collapsed_features = visual_features.view(batch_size, channels * height, width)

        # Step 4: Permute for RNN expectations -> (Sequence_Length, Batch, Features)
        sequence_input = collapsed_features.permute(2, 0, 1)

        # Step 5: Project features and process sequence
        projected_sequence = self.sequence_projection(sequence_input)
        rnn_output, _ = self.recurrent_layers(projected_sequence)

        # Step 6: Generate class predictions for each timestep
        class_predictions = self.classifier_head(rnn_output)

        return class_predictions

# Part 5: Sequence Decoding
**Prediction Selection**: Extracts the most probable character at every time step from the network's output.

**Sequence Cleaning**: Strips out blank tokens and collapses consecutive duplicate characters to form a clean prediction.

**Text Translation**: Converts the resulting numeric IDs back into a readable alphanumeric string.

In [ ]:
def decode_predictions(model_logits: torch.Tensor) -> List[str]:
    """
    Applies Greedy Decoding to the raw model outputs following CTC rules.

    Args:
        model_logits (torch.Tensor): Raw output from the CRNN.
                                     Expected shape: (Sequence_Length, Batch_Size, Total_Classes)

    Returns:
        List[str]: A list of decoded text strings for the batch.
    """
    # Find the index of the highest probability class at each timestep
    # We permute (1, 0) to change shape from (Seq_Len, Batch) to (Batch, Seq_Len)
    predicted_indices = torch.argmax(model_logits, dim=-1).permute(1, 0)

    batch_decoded_text = []

    # Iterate over each predicted sequence in the batch
    for sequence in predicted_indices:
        reconstructed_string = []
        previous_idx = None

        # Process each timestep token
        for current_idx in sequence:
            current_idx = current_idx.item()

            # CTC Rules:
            # 1. Ignore the 'blank' token (index 0)
            # 2. Ignore consecutive identical tokens
            if current_idx != 0 and current_idx != previous_idx:
                # Map the integer back to a character using our global idx_2_char dictionary
                reconstructed_string.append(idx_2_char[current_idx])

            # Update previous token for the next iteration's check
            previous_idx = current_idx

        # Join the list of characters into a single string
        batch_decoded_text.append("".join(reconstructed_string))

    return batch_decoded_text

# Part 6: Model Training Loop
**Forward Pass & Loss**: Feeds batches of images into the model and calculates the error by comparing the predicted sequence against the actual target lengths.

**Backpropagation**: Computes the gradients to figure out exactly how the network's internal weights need to change to minimize that error.

**Gradient Clipping**: Acts as a safety mechanism to cap the maximum size of the gradients, preventing unstable, massive updates that could derail training.

**Optimization**: Actively updates the model's weights based on the calculated gradients, officially "learning" from that batch of data.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    loss_fn: nn.Module,
    optimizer: Optimizer
) -> float:
    """
    Executes a single epoch of training using Connectionist Temporal Classification (CTC) loss.

    Args:
        model: The CRNN model (VisionSequenceReader).
        dataloader: PyTorch DataLoader providing batched training data.
        loss_fn: The loss function (nn.CTCLoss).
        optimizer: The optimization algorithm (e.g., Adam).

    Returns:
        float: The average training loss for this epoch.
    """
    # Set the model to training mode (enables Dropout and BatchNorm tracking)
    model.train()
    cumulative_loss = 0.0

    for batch_images, batch_targets, target_lengths in dataloader:
        # Transfer data to the designated hardware (GPU/CPU)
        batch_images = batch_images.to(config.compute_node)
        batch_targets = batch_targets.to(config.compute_node)

        # Clear previous gradients to prevent accumulation
        optimizer.zero_grad()

        # Forward pass: Generate predictions
        # Expected output shape: (Sequence_Length, Batch_Size, Total_Classes)
        model_predictions = model(batch_images)

        # CTC Loss requires the predicted sequence length for each item in the batch.
        # Since our images are resized to a fixed width, the prediction length is uniform.
        seq_length, batch_size, _ = model_predictions.size()
        prediction_lengths = torch.full(
            size=(batch_size,),
            fill_value=seq_length,
            dtype=torch.long
        ).to(config.compute_node)

        # Calculate CTC Loss
        # PyTorch CTCLoss explicitly requires log probabilities as input
        log_probabilities = model_predictions.log_softmax(dim=2)
        batch_loss = loss_fn(
            log_probabilities,
            batch_targets,
            prediction_lengths,
            target_lengths
        )

        # Backward pass: Compute gradients
        batch_loss.backward()

        # Apply Gradient Clipping to prevent exploding gradients (standard for LSTMs)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        # Update model weights
        optimizer.step()

        # Accumulate loss for performance tracking
        cumulative_loss += batch_loss.item()

    # Return the average loss across all batches
    average_epoch_loss = cumulative_loss / len(dataloader)
    return average_epoch_loss

# Part 7: Execution & Submission Pipeline
**Initialization**: Prepares the data loaders, initializes the model, and configures the loss function and optimizer.

**Training Loop**: Executes the training process across all epochs, continuously updating the model's weights to minimize error.

**Inference & Export**: Runs the unseen test images through the trained model, decodes the predictions, and saves them to a properly formatted CSV file.

In [ ]:
def execute_pipeline():
    """
    Main execution workflow: loads data, trains the OCR model,
    and generates a final submission CSV.
    """
    # 1. Validate Dataset Presence
    if not os.path.exists(train_csv_file):
        print(f"Error: Missing '{train_csv_file}'. Please verify your dataset path extraction.")
        return

    # 2. Load Metadata and Scan Test Files
    training_metadata = pd.read_csv(train_csv_file)

    valid_extensions = ('.png', '.jpg', '.jpeg')
    test_image_files = [file for file in os.listdir(test_image_folder) if file.lower().endswith(valid_extensions)]
    testing_metadata = pd.DataFrame({'image': test_image_files})

    print(f"Data Loaded: {len(training_metadata)} training samples, {len(testing_metadata)} test samples.")

    # 3. Initialize Datasets and DataLoaders
    training_dataset = TextRecognitionDataset(
        metadata_df=training_metadata,
        image_folder=train_image_folder,
        image_transform=training_augmentations
    )

    training_dataloader = DataLoader(
        dataset=training_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        collate_fn=ctc_collate_batch
    )

    testing_dataset = TextRecognitionDataset(
        metadata_df=testing_metadata,
        image_folder=test_image_folder,
        image_transform=validation_augmentations,
        evaluation_mode=True
    )

    testing_dataloader = DataLoader(
        dataset=testing_dataset,
        batch_size=config.batch_size,
        shuffle=False
    )

    # 4. Initialize Model, Loss Function, and Optimizer
    ocr_model = VisionSequenceReader(total_classes=total_model_classes).to(config.compute_node)

    # zero_infinity=True prevents NaN losses if the target length happens to exceed input length
    ctc_loss_criterion = nn.CTCLoss(blank=0, zero_infinity=True)

    optimizer = torch.optim.RMSprop(ocr_model.parameters(), lr=config.learning_rate)

    # 5. Training Loop
    print("Initiating model training phase...")
    for current_epoch in range(1, config.epochs + 1):
        avg_loss = train_one_epoch(
            model=ocr_model,
            dataloader=training_dataloader,
            loss_fn=ctc_loss_criterion,
            optimizer=optimizer
        )
        print(f"Epoch [{current_epoch}/{config.epochs}] -> Average Loss: {avg_loss:.4f}")

    # 6. Inference / Testing Phase
    ocr_model.eval()
    processed_images = []
    generated_predictions = []

    print("Generating predictions on the test dataset...")

    # Disable gradient calculation for faster inference
    with torch.no_grad():
        for batch_images, batch_filenames in testing_dataloader:
            batch_images = batch_images.to(config.compute_node)

            # Forward pass
            raw_outputs = ocr_model(batch_images)

            # Decode predictions
            text_predictions = decode_predictions(raw_outputs)

            processed_images.extend(batch_filenames)
            generated_predictions.extend(text_predictions)

    # 7. Save Submission File
    submission_record = pd.DataFrame({
        'image': processed_images,
        'prediction': generated_predictions
    })

    output_filename = "submission_VineetSinghania_24115162.csv"
    submission_record.to_csv(output_filename, index=False)
    print(f"Workflow complete. Submission saved as: {output_filename}")

# Execute the pipeline if the script is run directly
if __name__ == "__main__":
    execute_pipeline()

Data Loaded: 20000 training samples, 5000 test samples.
Initiating model training phase...
Epoch [1/30] -> Average Loss: 3.7641
Epoch [2/30] -> Average Loss: 3.5756
Epoch [3/30] -> Average Loss: 3.5734
Epoch [4/30] -> Average Loss: 3.5722
Epoch [5/30] -> Average Loss: 3.5694
Epoch [6/30] -> Average Loss: 3.4012
Epoch [7/30] -> Average Loss: 2.8905
Epoch [8/30] -> Average Loss: 2.2357
Epoch [9/30] -> Average Loss: 1.2687
Epoch [10/30] -> Average Loss: 0.5509
Epoch [11/30] -> Average Loss: 0.2018
Epoch [12/30] -> Average Loss: 0.0939
Epoch [13/30] -> Average Loss: 0.0524
Epoch [14/30] -> Average Loss: 0.0372
Epoch [15/30] -> Average Loss: 0.0257
Epoch [16/30] -> Average Loss: 0.0215
Epoch [17/30] -> Average Loss: 0.0158
Epoch [18/30] -> Average Loss: 0.0131
Epoch [19/30] -> Average Loss: 0.0122
Epoch [20/30] -> Average Loss: 0.0105
Epoch [21/30] -> Average Loss: 0.0080
Epoch [22/30] -> Average Loss: 0.0074
Epoch [23/30] -> Average Loss: 0.0147
Epoch [24/30] -> Average Loss: 0.0052
Epoch 